In [1]:
import cv2
import matplotlib.pyplot as plt
import scipy.io
import pandas as pd
import numpy as np
import os
import time
from PIL import Image
from matlab_import_helper import load_scr_matlab_data, print_data_summary, get_scope_data

# Settings

In [2]:
# Processing parameters
target_fps = 60 * 1000 / 1001  # Target frame rate (59.94 fps)
target_res = 16  # Target image resolution

original_video_fps = 120 * 1000 / 1001  # Original video frame rate
original_data_fps = 1000  # Original data sampling rate

smoothing_window = 1 / 20  # Running average window size for pressure signals(if enabled)
flag_smoothing = False  # Enable/disable signal smoothing

# Robot configuration: "1seg" for 1-segment, "2seg" for 2-segment robot
robot_type = "2seg"

# Set data paths based on robot type
if robot_type == "1seg":
    raw_video_path = "data/scr_match/smooth_input_rand_1_segment_15min_2Hz_max.mp4"
    raw_data_path = "data/scr_match/raw_data/smooth_input_rand_1_segment_15min_2Hz_max_pressure_data.mat"
elif robot_type == "2seg":
    raw_video_path = "data/scr_match/smooth_input_rand_2_segments_15min_2Hz_max_interpolated.mp4"
    raw_data_path = "data/scr_match/raw_data/smooth_input_rand_2_segments_15min_2Hz_max_interpolated_pressure_data.mat"


# Load and process video

In [3]:
# Load video file and extract frame information
cap = cv2.VideoCapture(raw_video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Create time and frame number arrays
frame_numbers = list(range(total_frames))
times = [i / fps for i in frame_numbers]

# Create DataFrame with frame timing information
df_video_frames = pd.DataFrame({
    "time": times,
    "frame_number": frame_numbers
})

cap.release()

# Reference frame for time synchronization (depressurization event)
# Values are obtained from manually inspecting the video and identifying the frame number of the depressurization to steady state
# Values given here are used in the paper
if robot_type == "1seg":
    depressure_frame_video = 108118
elif robot_type == "2seg":
    depressure_frame_video = 108257

depressure_time_video = df_video_frames.iloc[depressure_frame_video]["time"]

# Load and process pressure data

In [3]:
# Load MATLAB data file
data = load_scr_matlab_data(raw_data_path)
print_data_summary(data)


=== SCR MATLAB Data Summary ===

data_standard_struct:
  Time points: 953795
  Duration: 953.79 seconds
  Signals: ['p_is', 'p_set']
    p_is: shape=(953795, 6), range=[-987.378, 95589.851]
    p_set: shape=(953795, 6), range=[0.000, 88198.882]


In [4]:
# Extract pressure signals from MATLAB data
scope_time, scope_signals = get_scope_data(data)
p_is_data = scope_signals['p_is']

print(f"p_is data shape: {p_is_data.shape}")
print(f"Time points: {len(scope_time)}")

# Convert pressure from Pa to normalized units (relative to atmospheric pressure)
p_atmosphere = 101325  # Atmospheric pressure in Pa

# Create DataFrame with normalized pressure signals (p1-p6)
df_p_is = pd.DataFrame({
    'time': scope_time,
    'p1': p_is_data[:, 0] / p_atmosphere,
    'p2': p_is_data[:, 1] / p_atmosphere,
    'p3': p_is_data[:, 2] / p_atmosphere,
    'p4': p_is_data[:, 3] / p_atmosphere,
    'p5': p_is_data[:, 4] / p_atmosphere,
    'p6': p_is_data[:, 5] / p_atmosphere
})

print(df_p_is.head(20))

p_is data shape: (953795, 6)
Time points: 953795
     time        p1        p2        p3        p4        p5        p6
0   0.000  0.006853  0.010383  0.006437  0.002431  0.014316  0.014393
1   0.001  0.005191  0.013290  0.005814  0.003667  0.015246  0.013712
2   0.002  0.007891  0.010591  0.006230 -0.001908  0.012201  0.012661
3   0.003  0.003323  0.007268  0.005399 -0.001173  0.011325  0.011862
4   0.004  0.003946  0.009760  0.005191  0.000734  0.012083  0.013156
5   0.005  0.006437  0.010798  0.005399 -0.001218  0.012354  0.012584
6   0.006  0.008099  0.011214  0.005814  0.001223  0.014335  0.013952
7   0.007  0.004569  0.008929  0.005399  0.000089  0.012357  0.013584
8   0.008  0.006437  0.011214  0.006230 -0.000604  0.011894  0.014194
9   0.009  0.004776  0.011837  0.006022 -0.002106  0.010546  0.010469
10  0.010  0.004361  0.009552  0.005399  0.000559  0.012827  0.013594
11  0.011  0.006022  0.007060  0.005399  0.001862  0.013134  0.013671
12  0.012  0.005399  0.009760  0.005399  

In [5]:
# Apply smoothing to pressure signals (if enabled)
if flag_smoothing:
    average_window = int(1 / (1000 / 1001 / smoothing_window) * 1000)
    
    # Smooth all pressure channels using moving average
    smoothed_dict = {}
    for col in [f"p{i}" for i in range(1, 7)]:
        padded = np.pad(
            df_p_is[col].values,
            (average_window // 2, average_window - 1 - average_window // 2),
            mode='edge'
        )
        smoothed = np.convolve(
            padded,
            np.ones(average_window) / average_window,
            mode='valid'
        )
        smoothed_dict[col] = smoothed
    
    df_p_is_smooth = pd.DataFrame(smoothed_dict, index=df_p_is.index)
    df_p_is_smooth["time"] = df_p_is["time"].values

# Reference time for synchronization (depressurization event in data)
# Values are obtained from manually inspecting the pressure data and identifying the time of the depressurization to steady state
# Values given here are used in the paper
if robot_type == "1seg":
    depressure_time_data = 961.948
elif robot_type == "2seg":
    depressure_time_data = 969.210

# Select smoothed or raw data
if flag_smoothing:
    df_p_is_timecorr = df_p_is_smooth.copy()
else:
    df_p_is_timecorr = df_p_is.copy()

# Synchronize data time with video time using reference events
df_p_is_timecorr["time"] = df_p_is_timecorr["time"] - depressure_time_data + depressure_time_video

NameError: name 'depressure_time_video' is not defined

# Dataset generation

## Align frames and pressure data

In [7]:
# Generate dataset compatible with Latent_dynamics_learning script
print("Generating dataset compatible with Latent_dynamics_learning script...")

# Calculate frame sampling rate to achieve target fps
frame_sampling_rate = int(original_video_fps / target_fps)
print(f"Frame sampling rate: {frame_sampling_rate} (taking every {frame_sampling_rate}nd frame)")

# Find overlapping time range between video and pressure data
video_start_time = df_video_frames["time"].iloc[0]
video_end_time = df_video_frames["time"].iloc[-1]
data_start_time = df_p_is_timecorr["time"].iloc[0]
data_end_time = df_p_is_timecorr["time"].iloc[-1]

start_time = max(video_start_time, data_start_time)
end_time = min(video_end_time, data_end_time)

print(f"Video time range: {video_start_time:.2f} - {video_end_time:.2f} seconds")
print(f"Data time range: {data_start_time:.2f} - {data_end_time:.2f} seconds")
print(f"Using time range: {start_time:.2f} - {end_time:.2f} seconds")

# Filter video frames in the overlapping time range
video_mask = (df_video_frames["time"] >= start_time) & (df_video_frames["time"] <= end_time)
video_frames_in_range = df_video_frames[video_mask].copy()

# Sample frames at target frame rate
sampled_frames = video_frames_in_range.iloc[::frame_sampling_rate].copy()
print(f"Selected {len(sampled_frames)} frames for dataset")

# Filter pressure data in the same time range
data_mask = (df_p_is_timecorr["time"] >= start_time) & (df_p_is_timecorr["time"] <= end_time)
pressure_data_in_range = df_p_is_timecorr[data_mask].copy()

print(f"Pressure data points in range: {len(pressure_data_in_range)}")


Generating dataset compatible with Latent_dynamics_learning script...
Frame sampling rate: 2 (taking every 2nd frame)
Video time range: 0.00 - 914.91 seconds
Data time range: -32.90 - 920.89 seconds
Using time range: 0.00 - 914.91 seconds
Selected 54840 frames for dataset
Pressure data points in range: 914905


## Read video and extract frames

In [ ]:
# Extract and process video frames
print("Extracting and processing video frames...")

cap = cv2.VideoCapture(raw_video_path)
if not cap.isOpened():
    raise ValueError(f"Could not open video file: {raw_video_path}")

# Get video properties
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Calculate center square crop dimensions
min_dim = min(video_width, video_height)
crop_size = min_dim
crop_x = (video_width - crop_size) // 2
crop_y = (video_height - crop_size) // 2

print(f"Original video resolution: {video_width}x{video_height}")
print(f"Total video frames: {total_video_frames}")
print(f"Cropping center square: {crop_size}x{crop_size} from position ({crop_x}, {crop_y})")

# Get frame numbers and times for sampled frames
frame_numbers = sampled_frames["frame_number"].values
frame_times_list = sampled_frames["time"].values

needed_frames = set(frame_numbers)
print(f"Need to extract {len(needed_frames)} frames")

# Pre-allocate output array
n_frames = len(frame_numbers)
extracted_frames = np.zeros((n_frames, 3, target_res, target_res), dtype=np.float32)
frame_times = []

# Create mapping from frame number to output index
frame_to_idx = {frame_num: idx for idx, frame_num in enumerate(frame_numbers)}

# Read video sequentially and process only needed frames
current_frame = 0
processed_count = 0
total_needed = len(needed_frames)
start_time_extract = time.time()

print(f"Reading video sequentially...")
print(f"Progress: 0/{total_needed} frames processed (0.0%)")

progress_interval = max(1, total_needed // 100)
last_progress_time = start_time_extract

while current_frame < total_video_frames and cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    if current_frame in needed_frames:
        output_idx = frame_to_idx[current_frame]
        
        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # Crop center square and resize
        cropped = frame_rgb[crop_y:crop_y+crop_size, crop_x:crop_x+crop_size]
        resized = cv2.resize(cropped, (target_res, target_res), interpolation=cv2.INTER_AREA)
        
        # Normalize to [0, 1] and transpose to (C, H, W)
        normalized = (resized.astype(np.float32) / 255.0).transpose(2, 0, 1)
        
        extracted_frames[output_idx] = normalized
        frame_times.append((current_frame, frame_times_list[output_idx]))
        processed_count += 1
        
        # Progress reporting
        if processed_count % progress_interval == 0 or processed_count == total_needed:
            current_time = time.time()
            progress_pct = (processed_count / total_needed) * 100
            
            if processed_count > 0:
                frames_per_second = processed_count / (current_time - start_time_extract)
                remaining_frames = total_needed - processed_count
                eta_seconds = remaining_frames / frames_per_second if frames_per_second > 0 else 0
                eta_str = f"ETA: {eta_seconds:.1f}s" if eta_seconds > 0 else "ETA: calculating..."
            else:
                eta_str = "ETA: calculating..."
            
            print(f"Progress: {processed_count}/{total_needed} frames processed ({progress_pct:.1f}%) - {eta_str}")
            last_progress_time = current_time
    
    current_frame += 1

cap.release()

# Calculate timing statistics
end_time_extract = time.time()
total_time_extract = end_time_extract - start_time_extract
frames_per_second = processed_count / total_time_extract if total_time_extract > 0 else 0

print(f"\n=== EXTRACTION COMPLETE ===")
print(f"Successfully extracted {processed_count} frames")
print(f"Frame shape: {extracted_frames[0].shape}")
print(f"Total time: {total_time_extract:.2f} seconds")
print(f"Processing speed: {frames_per_second:.1f} frames/second")
print(f"Video reading speed: {current_frame / total_time_extract:.1f} frames/second (including skipped frames)")

images_array = extracted_frames
print(f"Images array shape: {images_array.shape}")

# Sort frame_times to match original order
sorted_indices = np.argsort(frame_numbers)
frame_times = [frame_times[i] for i in sorted_indices]


Extracting and processing video frames...
Original video resolution: 1920x1080
Total video frames: 109680
Cropping center square: 1080x1080 from position (420, 0)
Need to extract 54840 frames
Reading video sequentially...
Progress: 0/54840 frames processed (0.0%)
Progress: 548/54840 frames processed (1.0%) - ETA: 903.3s
Progress: 1096/54840 frames processed (2.0%) - ETA: 907.0s
Progress: 1644/54840 frames processed (3.0%) - ETA: 867.5s


## Calculating actuation pressure at video frames

In [ ]:
# Align pressure signals with extracted frames
print("Aligning actuation signals with extracted frames...")

# Interpolate pressure data to match frame times
pressure_signals = []
frame_times_array = np.stack(frame_times)[:, 1]

for i in range(1, 7):  # p1 to p6
    p_values = pressure_data_in_range[f"p{i}"].values
    p_times = pressure_data_in_range["time"].values
    interpolated_p = np.interp(frame_times_array, p_times, p_values)
    pressure_signals.append(interpolated_p)

# Convert to numpy array: shape (n_frames, 6)
pressure_array = np.column_stack(pressure_signals)
print(f"Pressure signals shape: {pressure_array.shape}")

# Check for NaN values
if np.any(np.isnan(pressure_array)):
    print("Warning: NaN values found in pressure data!")
    print(f"NaN count: {np.sum(np.isnan(pressure_array))}")
else:
    print("No NaN values found in pressure data")

# Check data ranges
print(f"\nPressure data ranges:")
for i in range(6):
    p_min, p_max = pressure_array[:, i].min(), pressure_array[:, i].max()
    print(f"p{i+1}: [{p_min:.4f}, {p_max:.4f}]")


## Create dataset dictionary

In [ ]:
# Create dataset in format compatible with Latent_dynamics_learning script
print("Creating dataset in compatible format...")

# Ensure we have the same number of frames and pressure samples
n_frames = len(images_array)
n_pressure_samples = len(pressure_array)

if n_frames != n_pressure_samples:
    print(f"Warning: Mismatch between frames ({n_frames}) and pressure samples ({n_pressure_samples})")
    min_length = min(n_frames, n_pressure_samples)
    images_array = images_array[:min_length]
    pressure_array = pressure_array[:min_length]
    print(f"Truncated to {min_length} samples")

# Create dataset dictionary with images
dataset_dict = {
    'images': images_array,  # Shape: (N, C, H, W)
}

# Add pressure signals based on robot type
# 1-segment robot uses 2 pressure channels, 2-segment uses 4
if robot_type == "1seg":
    p_indexes = [3, 5]
    for i in range(2):
        dataset_dict[f'p{i+1}'] = pressure_array[:, p_indexes[i]]
elif robot_type == "2seg":
    p_indexes = [0, 2, 3, 5]
    for i in range(4):
        dataset_dict[f'p{i+1}'] = pressure_array[:, p_indexes[i]]

print(f"Dataset created with {len(images_array)} samples")
print(f"Images shape: {images_array.shape}")
print(f"Pressure signals shape: {pressure_array.shape}")

# Verify the dataset structure
print("\nDataset structure:")
for key, value in dataset_dict.items():
    if isinstance(value, np.ndarray):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value)}")

# Check data quality
print(f"\nData quality checks:")
print(f"  Images range: [{images_array.min():.3f}, {images_array.max():.3f}]")
print(f"  Images mean: {images_array.mean():.3f}")
print(f"  Images std: {images_array.std():.3f}")

for i in range(6):
    p_values = pressure_array[:, i]
    print(f"  p{i+1} range: [{p_values.min():.4f}, {p_values.max():.4f}], mean: {p_values.mean():.4f}")


## Saving dataset

In [ ]:
# Save the dataset
print("Saving dataset...")

# Create output directory if it doesn't exist
output_dir = "data/scr_match"
os.makedirs(output_dir, exist_ok=True)

# Generate filename with parameters
dataset_filename = f"scr_dataset_raw_{robot_type}_{target_res}x{target_res}_{int(target_fps)}fps.npz"
dataset_path = os.path.join(output_dir, dataset_filename)

# Save the dataset
np.savez_compressed(dataset_path, **dataset_dict)

print(f"Dataset saved to: {dataset_path}")

# Verify the saved dataset
print("\nVerifying saved dataset...")
loaded_dataset = np.load(dataset_path)
print("Keys in saved dataset:", list(loaded_dataset.keys()))

for key in loaded_dataset.keys():
    data = loaded_dataset[key]
    print(f"  {key}: shape {data.shape}, dtype {data.dtype}")

# Calculate dataset size
file_size_mb = os.path.getsize(dataset_path) / (1024 * 1024)
print(f"\nDataset file size: {file_size_mb:.2f} MB")

# Summary
print(f"\n=== Dataset Generation Summary ===")
print(f"Total samples: {len(images_array)}")
print(f"Image resolution: {target_res}x{target_res}")
print(f"Target FPS: {target_fps:.2f}")
print(f"Time range: {start_time:.2f}s - {end_time:.2f}s")
print(f"Duration: {end_time - start_time:.2f} seconds")
print(f"Actual FPS: {len(images_array) / (end_time - start_time):.2f}")
print(f"Dataset saved to: {dataset_path}")

print(f"\nTo use this dataset in Latent_dynamics_learning.ipynb:")
print(f"1. Update the config to set 'dataset': 'scr_match'")
print(f"2. Update the datasets dictionary to include:")
print(f"   'scr_match': f'scr_match/{dataset_filename}'")
print(f"3. Set 'actuation_dim': 6  # for p1-p6 signals")
print(f"4. Set 'resolution': {target_res}  # or 32 if you want to resize further")
